# RAG

In [ ]:
# Librerías estándar
import os

# LangChain
from langchain_ollama import OllamaLLM
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


## 1. Configuración del modelo LLM

In [ ]:
llm = OllamaLLM(model="llama3.2:1b")

## 2. Carga de documentos

In [ ]:
# Definimos la ruta de la carpeta
pdf_folder_path = "./data/"
docs = []

# Iteramos sobre los archivos en la carpeta 'data'
for file in os.listdir(pdf_folder_path):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder_path, file))
        docs.extend(loader.load())

print(f"✅ Se cargaron {len(docs)} páginas de los PDFs.")

## 3. División en fragmentos

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100, # Solapamiento para no perder contexto entre trozos
    length_function=len
)

splits = text_splitter.split_documents(docs)
print(f"✅ Los PDFs se dividieron en {len(splits)} fragmentos.")

In [ ]:
# Ver el contenido del primer fragmento
if splits:
    print("Muestra del primer fragmento:")
    print(splits[0].page_content)

## 4. Creación de embeddings

In [ ]:
# Definimos el modelo de embeddings
print("🔄 Cargando modelo de embeddings...")
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

## 5. Almacén vectorial FAISS

In [ ]:
# 2. Creamos el almacén vectorial (Vector Store)
# Esto toma tus 'splits' de la Fase 3 y los convierte en vectores usando el modelo
print("🧠 Generando vectores e indexando en FAISS (esto puede tardar unos segundos)...")
vectorstore = FAISS.from_documents(splits, embeddings_model)

print("✅ Almacén vectorial creado exitosamente.")

# 3. Prueba rápida de búsqueda semántica (Retriever)
query = "¿Qué objeto encontró la NASA en las Marianas?"
docs_relacionados = vectorstore.similarity_search(query, k=2)

print("\n🔍 Prueba de búsqueda:")
for i, doc in enumerate(docs_relacionados):
    print(f"Resultado {i+1}: {doc.page_content[:100]}...")

## 6. Definición del pipeline RAG

In [ ]:
# 1. Definimos el Prompt
template = """Eres un asistente técnico experto. Responde la pregunta basándote únicamente en el siguiente contexto:
{context}

Pregunta: {question}

Respuesta:"""

prompt = ChatPromptTemplate.from_template(template)

# 2. Configuramos el flujo (Chain)
# Este pipeline: toma la pregunta -> busca en FAISS -> llena el prompt -> le pregunta a Llama
rag_chain = (
    {"context": vectorstore.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 7. Prueba final

In [ ]:
pregunta_final = "¿Cuales son los cuatro roles identificados en la colonia de Towada?"
print(f"Preguntando: {pregunta_final}\n")

respuesta_final = rag_chain.invoke(pregunta_final)
print("--- RESPUESTA DEL RAG ---")
print(respuesta_final)